# mu-logsigma-encoder-head — worked example 1: Split encoder head using einops Rearrange instead of chunk

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mu-logsigma-encoder-head`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

A VAE encoder head projects features to `2 * latent_dim` and then splits into `(mu, logsigma)`. Instead of `chunk(2, dim=-1)`, you can use einops `Rearrange('b (n d) -> n b d', n=2)` to reshape the output into a `(2, B, latent_dim)` tensor and unpack along axis 0. Both approaches are equivalent; the einops form is more explicit about the semantics of the split.

## Worked solution

**Step 1 — apply the double-width linear projection.** `features @ weight.T + bias` maps `(B, D)` to `(B, 2 * latent_dim)`. This is the standard VAE encoder head projection.

**Step 2 — rearrange with einops.** `Rearrange('b (n d) -> n b d', n=2)` interprets the last dimension as `2 * latent_dim = n * d` (with `n=2`) and reshapes to `(2, B, latent_dim)`. The result stacks `mu` and `logsigma` along axis 0.

**Step 3 — unpack.** Indexing `params_3d[0]` gives `mu` of shape `(B, latent_dim)` and `params_3d[1]` gives `logsigma` of shape `(B, latent_dim)`.

**Why know both forms?** `chunk` is simpler for a single split, but `Rearrange` generalises: if you wanted to split into K heads (e.g., multi-head attention), the same pattern works with `n=K`.

In [ ]:
import torch
from einops.layers.torch import Rearrange
from einops import rearrange

torch.manual_seed(0)

def encoder_head_rearrange(features: torch.Tensor, weight: torch.Tensor,
                           bias: torch.Tensor, latent_dim: int):
    """
    Project features to (mu, logsigma) using Rearrange instead of chunk.
    features: (B, D)
    weight:   (2*latent_dim, D)
    bias:     (2*latent_dim,)
    Returns: (mu, logsigma) each of shape (B, latent_dim)
    """
    params = features @ weight.T + bias              # (B, 2*latent_dim)
    # Rearrange: treat last dim as 2 groups of latent_dim
    params_3d = rearrange(params, 'b (n d) -> n b d', n=2)  # (2, B, latent_dim)
    mu       = params_3d[0]  # (B, latent_dim)
    logsigma = params_3d[1]  # (B, latent_dim)
    return mu, logsigma

# Exercise: verify equivalent to chunk
torch.manual_seed(7)
B, D, L = 5, 32, 8
features = torch.randn(B, D)
weight   = torch.randn(2 * L, D)
bias     = torch.randn(2 * L)

mu_re, ls_re = encoder_head_rearrange(features, weight, bias, L)

# Compare to chunk reference
params = features @ weight.T + bias
mu_ck, ls_ck = params.chunk(2, dim=-1)

assert torch.allclose(mu_re,  mu_ck,  atol=1e-6), "mu mismatch"
assert torch.allclose(ls_re, ls_ck, atol=1e-6), "logsigma mismatch"
print(f"mu shape: {mu_re.shape}, logsigma shape: {ls_re.shape}")
print("Rearrange and chunk produce identical results!")